# Gabarito dos Exercicios - Modulo 4

**Tema:** EDA, qualidade dos dados e conclusoes analiticas

> Arquivo reservado apenas para as respostas dos exercicios.

In [ ]:
from pathlib import Path
import io
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 110

def carregar_brazilian_cities(caminho):
    with open(caminho, 'r', encoding='utf-8', newline='') as arquivo:
        bruto = arquivo.read()

    linhas = bruto.split('\r\n')

    def corrigir(linha):
        if linha.startswith('"') and linha.endswith('"'):
            linha = linha[1:-1]
        return linha.replace('""', '"')

    conteudo = '\n'.join(corrigir(linha) for linha in linhas if linha.strip())
    return pd.read_csv(io.StringIO(conteudo))

caminhos = [
    Path('../../modulo-3/datasets/brazilian_city.csv'),
    Path('../datasets/brazilian_city.csv'),
    Path('../../projeto-final/datasets/brazilian_city.csv'),
]
caminho_dataset = next(caminho for caminho in caminhos if caminho.exists())
df = carregar_brazilian_cities(caminho_dataset)

df = df.rename(columns={
    'IDHM Ranking 2010': 'IDHM_Ranking',
    'IBGE_CROP_PRODUCTION_$': 'IBGE_CROP_PROD',
    'WAL-MART': 'WALMART',
    'IBGE_1-4': 'IBGE_1a4',
    'IBGE_5-9': 'IBGE_5a9',
    'IBGE_10-14': 'IBGE_10a14',
    'IBGE_15-59': 'IBGE_15a59',
    'IBGE_60+': 'IBGE_60mais',
})

regioes = {
    'Norte': ['AC', 'AP', 'AM', 'PA', 'RO', 'RR', 'TO'],
    'Nordeste': ['AL', 'BA', 'CE', 'MA', 'PB', 'PE', 'PI', 'RN', 'SE'],
    'Centro-Oeste': ['DF', 'GO', 'MT', 'MS'],
    'Sudeste': ['ES', 'MG', 'RJ', 'SP'],
    'Sul': ['PR', 'RS', 'SC'],
}

df['REGIAO'] = df['STATE'].map({uf: regiao for regiao, ufs in regioes.items() for uf in ufs})
df['POP_FINAL'] = df['IBGE_RES_POP'].where(df['IBGE_RES_POP'] > 0, df['ESTIMATED_POP'])
df['DENSIDADE'] = np.where(df['AREA'] > 0, df['POP_FINAL'] / df['AREA'], np.nan)
df['TIPO'] = df['CAPITAL'].map({1: 'Capital', 0: 'Interior'})
df['IDHM_FAIXA'] = pd.cut(
    df['IDHM'],
    bins=[0, 0.499, 0.599, 0.699, 0.799, 1],
    labels=['Muito baixo', 'Baixo', 'Medio', 'Alto', 'Muito alto'],
)

df.head()

## Exercicio 1 - Estrutura e qualidade dos dados

In [ ]:
resumo_qualidade = pd.DataFrame({
    'coluna': df.columns,
    'tipo': df.dtypes.astype(str).values,
    'nulos': df.isna().sum().values,
    'nulos_%': (df.isna().mean().values * 100).round(2),
    'unicos': df.nunique(dropna=True).values,
})

resumo_qualidade.sort_values('nulos_%', ascending=False).head(15)

**Resposta:** o dataset possui dados municipais reais, com variaveis demograficas, economicas, territoriais e sociais. A verificacao de nulos mostra quais colunas exigem maior cuidado antes da analise.

## Exercicio 2 - Estatisticas descritivas

In [ ]:
variaveis = ['IDHM', 'GDP_CAPITA', 'POP_FINAL', 'AREA', 'DENSIDADE']
df[variaveis].describe().T.round(2)

**Resposta:** `POP_FINAL`, `AREA`, `DENSIDADE` e `GDP_CAPITA` apresentam alta dispersao. Isso indica presenca de municipios muito diferentes entre si e justifica o uso de mediana, quartis e graficos com escala logaritmica.

## Exercicio 3 - Distribuicao do IDHM

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df['IDHM'].dropna(), bins=35, kde=True, color='#4C72B0', ax=ax)
ax.axvline(df['IDHM'].mean(), color='#C44E52', linestyle='--', label=f"Media: {df['IDHM'].mean():.3f}")
ax.axvline(df['IDHM'].median(), color='#55A868', linestyle='-.', label=f"Mediana: {df['IDHM'].median():.3f}")
ax.set_title('Distribuicao do IDHM municipal')
ax.set_xlabel('IDHM')
ax.set_ylabel('Quantidade de municipios')
ax.legend()
plt.tight_layout()
plt.show()

**Resposta:** a maior parte dos municipios se concentra em faixas intermediarias de IDHM. Valores muito baixos ou muito altos existem, mas sao menos frequentes.

## Exercicio 4 - Comparacao por regiao

In [ ]:
resumo_regiao = df.groupby('REGIAO').agg(
    municipios=('CITY', 'count'),
    idhm_medio=('IDHM', 'mean'),
    idhm_mediano=('IDHM', 'median'),
    pib_pc_mediano=('GDP_CAPITA', 'median'),
    populacao_total=('POP_FINAL', 'sum'),
).sort_values('idhm_mediano', ascending=False)

resumo_regiao.round(2)

In [ ]:
ordem = resumo_regiao.index.tolist()

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x='REGIAO', y='IDHM', order=ordem, ax=ax)
ax.set_title('IDHM por regiao brasileira')
ax.set_xlabel('Regiao')
ax.set_ylabel('IDHM')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

**Resposta:** a comparacao regional evidencia desigualdades territoriais. As regioes com maior mediana de IDHM tendem a concentrar melhores indicadores economicos e sociais.

## Exercicio 5 - Relacao entre PIB per capita e IDHM

In [ ]:
base_scatter = df[(df['GDP_CAPITA'] > 0) & df['IDHM'].notna() & df['REGIAO'].notna()].copy()

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=base_scatter,
    x='GDP_CAPITA',
    y='IDHM',
    hue='REGIAO',
    alpha=0.65,
    s=35,
    ax=ax,
)
ax.set_xscale('log')
ax.set_title('PIB per capita e IDHM dos municipios')
ax.set_xlabel('PIB per capita em escala log')
ax.set_ylabel('IDHM')
ax.legend(title='Regiao', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

base_scatter[['GDP_CAPITA', 'IDHM']].corr().iloc[0, 1].round(3)

**Resposta:** existe relacao positiva entre PIB per capita e IDHM, mas ela nao e perfeita. Desenvolvimento humano depende tambem de educacao, saude, infraestrutura e distribuicao das oportunidades.

## Exercicio 6 - Conclusao final

**Resposta final:** a EDA mostra que os municipios brasileiros sao bastante heterogeneos. O IDHM varia de forma relevante entre regioes, o PIB per capita apresenta forte assimetria e a populacao se concentra em poucos municipios. A analise tambem indica que renda ajuda a explicar desenvolvimento humano, mas nao explica tudo. Para decisoes publicas ou de negocio, e recomendavel cruzar indicadores sociais, economicos e demograficos antes de definir prioridades.